In [15]:
%pip install pykrx pandas
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [61]:
import os

import time
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(".env", override=True)
from pykrx import stock
from dotenv import load_dotenv

load_dotenv()

True

In [19]:
# kospi index
def collect_index_future_returns(
    start_date: str,
    end_date: str,
    ticker: str,
    index_name: str,
    output_path: str,
):
    """지수 종가와 t+1~t+5 거래일의 미래 종가 수익률을 Excel로 저장합니다."""
    df = stock.get_index_ohlcv(
        start_date, end_date, ticker, name_display=False
    )

    if df.empty or '종가' not in df.columns:
        raise RuntimeError(f'{index_name} 지수 데이터를 받지 못했습니다.')

    result = pd.DataFrame(index=df.index)
    result['Index_Code'] = ticker
    result['Index_Name'] = index_name
    result['Close'] = df['종가']
    result['Daily_Return'] = result['Close'].pct_change()

    for horizon in range(1, 6):
        future_close = result['Close'].shift(-horizon)
        result[f'Close_t_plus_{horizon}'] = future_close
        result[f'Future_Return_t_plus_{horizon}'] = (
            future_close / result['Close'] - 1
        )

    result.index.name = 'Date'
    result.reset_index(inplace=True)
    result.to_excel(output_path, index=False, engine='openpyxl')
    print(f'{index_name} 지수 수익률 저장 완료: {output_path} ({len(result)}일)')
    return result


kospi_index_returns = collect_index_future_returns(
    start_date='20210701',
    end_date='20260630',
    ticker='1001',
    index_name='KOSPI',
    output_path='./kospi_index_returns.xlsx',
)

KRX 세션 만료, 재로그인 시도...
KRX 세션 갱신 완료.
KOSPI 지수 수익률 저장 완료: ./kospi_index_returns.xlsx (1222일)


In [4]:
# kosdaq index
kosdaq_index_returns = collect_index_future_returns(
    start_date='20210701',
    end_date='20260630',
    ticker='2001',
    index_name='KOSDAQ',
    output_path='./kosdaq_index_returns.xlsx',
)

KOSDAQ 지수 수익률 저장 완료: ./kosdaq_index_returns.xlsx (1222일)


In [3]:
# kospi by ticker

def collect_kospi_daily_data(start_date: str, end_date: str, directory: str = "./data_kospi"):
    """KOSPI 시장의 일별 OHLCV 및 개인/기관/외국인 순매수 데이터 수집"""
    os.makedirs(directory, exist_ok=True)
    
    # 영업일 리스트 추출
    sample = stock.get_index_ohlcv(start_date, end_date, "1001", name_display=False)
    trading_days = sample.index.strftime("%Y%m%d").tolist()
    
    print(f"--- KOSPI 데이터 수집 시작 (총 {len(trading_days)}일) ---")
    
    for date in trading_days:
        file_path = os.path.join(directory, f"kospi_{date}.xlsx")
        if os.path.exists(file_path):
            continue
            
        try:
            # 1. KOSPI 전 종목 OHLCV 조회
            df_kospi = stock.get_market_ohlcv(date, market="KOSPI")
            
            # 2. 개인 및 전체 투자자의 종목별 매수·매도 주식 수 조회
            investor_prefixes = {
                '개인': 'Retail',
                '기관합계': 'Institution',
                '외국인': 'Foreign',
                '전체': 'Total',
            }
            volume_frames = []
            for investor, prefix in investor_prefixes.items():
                investor_df = stock.get_market_net_purchases_of_equities(
                    date, date, market="KOSPI", investor=investor
                )
                volume_frames.append(
                    investor_df[['매수거래량', '매도거래량']].rename(columns={
                        '매수거래량': f'{prefix}_Buy_Volume',
                        '매도거래량': f'{prefix}_Sell_Volume',
                    })
                )
            df_investor_volume = (
                pd.concat(volume_frames, axis=1).fillna(0).astype('int64')
            )
            
            # 3. 종목코드 인덱스를 기준으로 OHLCV와 투자자 거래량 병합
            df_merged = df_kospi.join(df_investor_volume, how='inner')
            
            # 4. 컬럼 정리 및 저장
            df_merged.insert(0, 'Date', date)
            df_merged.index.name = 'Ticker'
            df_merged.reset_index(inplace=True)
            
            df_merged.to_excel(file_path, index=False, engine='openpyxl')
            print(f"[KOSPI] {date} 수집 완료")
            time.sleep(0.5) # KRX 서버 차단 방지
            
        except Exception as e:
            print(f"[KOSPI 오류] {date}: {e}")

In [4]:
# kosdaq by ticker
def collect_kosdaq_daily_data(start_date: str, end_date: str, directory: str = "./data_kosdaq"):
    """KOSDAQ 시장의 일별 OHLCV 및 개인/기관/외국인 순매수 데이터 수집"""
    os.makedirs(directory, exist_ok=True)
    
    # 영업일 리스트 추출
    sample = stock.get_index_ohlcv(start_date, end_date, "2001", name_display=False)
    trading_days = sample.index.strftime("%Y%m%d").tolist()
    
    print(f"\n--- KOSDAQ 데이터 수집 시작 (총 {len(trading_days)}일) ---")
    
    for date in trading_days:
        file_path = os.path.join(directory, f"kosdaq_{date}.xlsx")
        if os.path.exists(file_path):
            continue
            
        try:
            # 1. KOSDAQ 전 종목 OHLCV 조회
            df_kosdaq = stock.get_market_ohlcv(date, market="KOSDAQ")
            
            # 2. 개인 및 전체 투자자의 종목별 매수·매도 주식 수 조회
            investor_prefixes = {
                '개인': 'Retail',
                '기관합계': 'Institution',
                '외국인': 'Foreign',
                '전체': 'Total',
            }
            volume_frames = []
            for investor, prefix in investor_prefixes.items():
                investor_df = stock.get_market_net_purchases_of_equities(
                    date, date, market="KOSDAQ", investor=investor
                )
                volume_frames.append(
                    investor_df[['매수거래량', '매도거래량']].rename(columns={
                        '매수거래량': f'{prefix}_Buy_Volume',
                        '매도거래량': f'{prefix}_Sell_Volume',
                    })
                )
            df_investor_volume = (
                pd.concat(volume_frames, axis=1).fillna(0).astype('int64')
            )
            
            # 3. 종목코드 인덱스를 기준으로 OHLCV와 투자자 거래량 병합
            df_merged = df_kosdaq.join(df_investor_volume, how='inner')
            
            # 4. 컬럼 정리 및 저장
            df_merged.insert(0, 'Date', date)
            df_merged.index.name = 'Ticker'
            df_merged.reset_index(inplace=True)
            
            df_merged.to_excel(file_path, index=False, engine='openpyxl')
            print(f"[KOSDAQ] {date} 수집 완료")
            time.sleep(0.5) # KRX 서버 차단 방지
            
        except Exception as e:
            print(f"[KOSDAQ 오류] {date}: {e}")

In [122]:
if __name__ == "__main__":
    START = "20180327"
    END = "20180327"
    
    # KOSPI 수집
    collect_kospi_daily_data(START, END, directory="./data_kospi")
    
    # KOSDAQ 수집
    collect_kosdaq_daily_data(START, END, directory="./data_kosdaq")

--- KOSPI 데이터 수집 시작 (총 1일) ---
[KOSPI] 20180327 수집 완료

--- KOSDAQ 데이터 수집 시작 (총 1일) ---


In [ ]:

import glob
import os
import pandas as pd

def normalize_ticker(value):
    if pd.isna(value):
        return ""
    s = str(value).strip()
    return s.zfill(6) if s.isdigit() else s

for market_dir in ["./data_kospi", "./data_kosdaq"]:
    file_paths = sorted(
        glob.glob(os.path.join(market_dir, "*.xlsx")),
        key=lambda x: os.path.basename(x),
    )

    prev_df = None
    prev_path = None

    for file_path in file_paths:
        if os.path.basename(file_path).startswith("~$"):
            continue

        try:
            df = pd.read_excel(file_path, engine="openpyxl")
        except Exception as e:
            print(f"[SKIP] {file_path}: {type(e).__name__}: {e}")
            continue

        if "Ticker" not in df.columns:
            continue

        df = df.copy()
        df["Ticker"] = df["Ticker"].map(normalize_ticker)

        if prev_df is not None:
            prev_tickers = set(prev_df["Ticker"].dropna())
            curr_tickers = set(df["Ticker"].dropna())
            inconsistent_tickers = prev_tickers.symmetric_difference(curr_tickers)

            if inconsistent_tickers:
                prev_df = prev_df[~prev_df["Ticker"].isin(inconsistent_tickers)].copy()
                df = df[~df["Ticker"].isin(inconsistent_tickers)].copy()

                prev_df.to_excel(prev_path, index=False, engine="openpyxl")
                df.to_excel(file_path, index=False, engine="openpyxl")

                print(
                    f"[{market_dir}] removed {len(inconsistent_tickers)} inconsistent tickers: "
                    f"{sorted(inconsistent_tickers)[:10]}"
                )

        prev_df = df.copy()
        prev_path = file_path


In [ ]:
# Table 1 규격의 CNN 입력 이미지 생성 (OHLC / OHLC+Volume, 선택적으로 MA 포함)
# Pillow가 없다면 먼저 `%pip install pillow`를 한 번 실행하세요.
from pathlib import Path
import glob
import numpy as np
import pandas as pd
from PIL import Image

# ------------------------------ 설정 ------------------------------
IMAGE_SPECS = {
    # n_days: (width, total_height, price_height, volume_height)
    5:  (15, 32, 25, 6),
    20: (60, 64, 51, 12),
    60: (180, 96, 76, 19),
}
MARKET_DIRS = {"KOSPI": Path("./data_kospi"), "KOSDAQ": Path("./data_kosdaq")}
OUTPUT_ROOT = Path("./krx_chart_images")
INCLUDE_MOVING_AVERAGE = True  # Figure 4처럼 최종 이미지에 n일 이동평균선 포함
WINDOW_STEP = 1                 # 1이면 가능한 모든 종료 거래일을 생성
START_END_DATE = None           # 예: ("20210101", "20231231"), 전체 기간은 None
OVERWRITE = False

PRICE_COLUMNS = ["시가", "고가", "저가", "종가"]
REQUIRED_COLUMNS = ["Date", "Ticker", *PRICE_COLUMNS, "거래량"]


def _ticker6(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().split(".")[0]
    return value.zfill(6) if value.isdigit() else value


def load_market_history(market_dir):
    """일별 Excel 파일을 종목-날짜 패널로 결합합니다."""
    paths = [p for p in sorted(glob.glob(str(Path(market_dir) / "*.xlsx")))
             if not Path(p).name.startswith("~$")]
    frames = []
    for path in paths:
        try:
            frame = pd.read_excel(path, usecols=lambda c: c in REQUIRED_COLUMNS, engine="openpyxl")
        except Exception as exc:
            print(f"[SKIP] {path}: {type(exc).__name__}: {exc}")
            continue
        missing = set(REQUIRED_COLUMNS) - set(frame.columns)
        if missing:
            print(f"[SKIP] {path}: missing {sorted(missing)}")
            continue
        frames.append(frame[REQUIRED_COLUMNS])

    if not frames:
        return pd.DataFrame(columns=REQUIRED_COLUMNS)

    history = pd.concat(frames, ignore_index=True)
    history["Date"] = pd.to_datetime(history["Date"].astype(str), errors="coerce")
    history["Ticker"] = history["Ticker"].map(_ticker6)
    for column in [*PRICE_COLUMNS, "거래량"]:
        history[column] = pd.to_numeric(history[column], errors="coerce")
    return (history.dropna(subset=["Date"])
                   .drop_duplicates(["Ticker", "Date"], keep="last")
                   .sort_values(["Ticker", "Date"]))


def _scale_to_rows(values, height, data_min, data_max):
    """가격을 영상 행 번호로 변환합니다(최댓값=맨 위, 최솟값=맨 아래)."""
    values = np.asarray(values, dtype=float)
    rows = np.full(values.shape, -1, dtype=np.int32)
    valid = np.isfinite(values)
    if not valid.any():
        return rows
    if not np.isfinite(data_min) or not np.isfinite(data_max) or data_max <= data_min:
        rows[valid] = (height - 1) // 2
    else:
        scaled = (data_max - values[valid]) / (data_max - data_min) * (height - 1)
        rows[valid] = np.clip(np.rint(scaled), 0, height - 1).astype(np.int32)
    return rows


def _draw_line(canvas, x0, y0, x1, y1):
    """안티앨리어싱 없는 1픽셀 Bresenham 선을 그립니다."""
    dx, dy = abs(x1 - x0), -abs(y1 - y0)
    sx, sy = (1 if x0 < x1 else -1), (1 if y0 < y1 else -1)
    error = dx + dy
    while True:
        if 0 <= y0 < canvas.shape[0] and 0 <= x0 < canvas.shape[1]:
            canvas[y0, x0] = 255
        if x0 == x1 and y0 == y1:
            break
        twice = 2 * error
        if twice >= dy:
            error += dy
            x0 += sx
        if twice <= dx:
            error += dx
            y0 += sy


def render_market_image(window, n_days, with_volume, with_ma=True):
    """논문의 3픽셀 OHLC 표기법으로 uint8 흑백 영상을 반환합니다."""
    width, total_h, price_h, volume_h = IMAGE_SPECS[n_days]
    draw_price_h = price_h if with_volume else total_h
    image = np.zeros((total_h, width), dtype=np.uint8)

    # 세로축은 논문처럼 해당 창의 OHLC 최저/최고가에 맞춥니다.
    lows = window["저가"].to_numpy(float)
    highs = window["고가"].to_numpy(float)
    finite_low, finite_high = lows[np.isfinite(lows)], highs[np.isfinite(highs)]
    if not len(finite_low) or not len(finite_high):
        return image
    p_min, p_max = finite_low.min(), finite_high.max()
    rows = {c: _scale_to_rows(window[c], draw_price_h, p_min, p_max) for c in PRICE_COLUMNS}

    for day in range(n_days):
        x = 3 * day + 1
        o, h, l, c = (rows[col][day] for col in PRICE_COLUMNS)
        # high/low 중 하나라도 없으면 논문 각주처럼 OHLC bar 전체를 비웁니다.
        if h >= 0 and l >= 0:
            image[min(h, l):max(h, l) + 1, x] = 255
            if o >= 0:
                image[o, x - 1:x + 1] = 255       # 시가: 왼쪽 표시
            if c >= 0:
                image[c, x:x + 2] = 255           # 종가: 오른쪽 표시

    if with_ma and "MA" in window:
        ma_rows = _scale_to_rows(window["MA"], draw_price_h, p_min, p_max)
        previous = None
        for day, row in enumerate(ma_rows):
            if row < 0:
                previous = None
                continue
            point = (3 * day + 1, int(row))
            if previous is None:
                image[point[1], point[0]] = 255
            else:
                _draw_line(image, previous[0], previous[1], point[0], point[1])
            previous = point

    if with_volume:
        volumes = window["거래량"].to_numpy(float)
        valid = np.isfinite(volumes) & (volumes >= 0)
        vmax = volumes[valid].max() if valid.any() else 0
        base = total_h - 1
        for day, volume in enumerate(volumes):
            if not np.isfinite(volume) or volume < 0 or vmax <= 0:
                continue
            bar_h = max(1, int(np.rint(volume / vmax * volume_h))) if volume > 0 else 0
            if bar_h:
                image[base - bar_h + 1:base + 1, 3 * day + 1] = 255
        # price 영역과 volume 영역 사이는 항상 1픽셀 검정 행입니다.

    return image


def generate_table1_images(market_name, history, output_root=OUTPUT_ROOT):
    """한 시장의 모든 종목/rolling window에 대해 Table 1의 6종 이미지를 저장합니다."""
    if history.empty:
        print(f"[{market_name}] 생성할 데이터가 없습니다.")
        return {"saved": 0, "skipped": 0}

    calendar = pd.DatetimeIndex(sorted(history["Date"].unique()))
    date_bounds = None if START_END_DATE is None else tuple(pd.to_datetime(x) for x in START_END_DATE)
    stats = {"saved": 0, "skipped": 0}

    for ticker, ticker_df in history.groupby("Ticker", sort=True):
        series = ticker_df.set_index("Date")[PRICE_COLUMNS + ["거래량"]].reindex(calendar)
        # 각 n일 MA는 이미지 창 이전의 가격도 이용해 먼저 계산합니다.
        for n_days, (width, total_h, _, _) in IMAGE_SPECS.items():
            work = series.copy()
            work["MA"] = work["종가"].rolling(n_days, min_periods=n_days).mean()
            for end_pos in range(n_days - 1, len(work), WINDOW_STEP):
                end_date = calendar[end_pos]
                if date_bounds and not (date_bounds[0] <= end_date <= date_bounds[1]):
                    continue
                window = work.iloc[end_pos - n_days + 1:end_pos + 1]
                # IPO/상장폐지 창은 제외하되 중간 결측일은 검정(빈 칸)으로 허용합니다.
                if window.iloc[[0, -1]][PRICE_COLUMNS].isna().all(axis=1).any():
                    stats["skipped"] += 1
                    continue
                date_text = end_date.strftime("%Y%m%d")
                for kind, with_volume in (("OHLC", False), ("OHLCV", True)):
                    ma_tag = "_MA" if INCLUDE_MOVING_AVERAGE else ""
                    folder = Path(output_root) / market_name / f"{kind}{ma_tag}_{n_days}d_{width}x{total_h}"
                    folder.mkdir(parents=True, exist_ok=True)
                    target = folder / f"{ticker}_{date_text}.png"
                    if target.exists() and not OVERWRITE:
                        stats["skipped"] += 1
                        continue
                    pixels = render_market_image(window, n_days, with_volume, INCLUDE_MOVING_AVERAGE)
                    Image.fromarray(pixels, mode="L").save(target, optimize=True)
                    stats["saved"] += 1
        if stats["saved"] and stats["saved"] % 10000 == 0:
            print(f"[{market_name}] {stats['saved']:,}장 저장...")

    print(f"[{market_name}] 완료: 저장 {stats['saved']:,}, 건너뜀 {stats['skipped']:,}")
    return stats


# 실행: Table 1의 OHLC/OHLC+V × 5/20/60일 = 시장별 6종 폴더 생성
generation_stats = {}
for market_name, market_dir in MARKET_DIRS.items():
    market_history = load_market_history(market_dir)
    generation_stats[market_name] = generate_table1_images(market_name, market_history)
    del market_history
generation_stats
